# E5 step 2 · Per-image pixel counts for R11, R11s7, R9 and R1 on the 645 validation images

Task **E5** in `Essential_task_delegation/Next_Steps_Roadmap.md` (built 26 Oct 2026). GPU T4, about 1 hour.

**Why**
- Every O1 score was measured on all 645 validation images, and some of them leak (E5). A score without the leaked pairs needs each image's own counts, and no D4 run saved them.
- This run saves them once. Then the clean score for any list (the PC check's 21, the audit's 40, step 1's hash check) takes seconds on your PC, with no further GPU run.

**What it does**
- Rebuilds the D4 split exactly (R11's cells, verbatim) and runs each model as the D4 notebooks did: full size, padded to a multiple of 32, standard and 8-view TTA.
  - R11 and R11s7 read the after-image and the before-image (6 channels);
  - R9 and R1 read the after-image only.
- Saves a 4 × 4 pixel count (truth × prediction) for each image, model and mode.
- Checks that the sums reproduce each model's D4 CSV, and previews the scores without the PC check's 21 pairs and without all 86 fire images.

**Expectation, fixed before the run (26 Oct)**
- Every model reproduces its D4 CSV to within 0.0005, for mIoU and Destroyed IoU, standard and with TTA.
- Without the 21 pairs, Destroyed IoU drops by about 0.008–0.010, as R16c's did (0.3090 → 0.3000 on average).

**How to run on Kaggle** (account 2)
1. Settings:
   - GPU T4 (×1 or ×2), Internet On;
   - secret `HF_TOKEN`: the checkpoints are in the private repos `AbrarAlam/disasterm3-d4-R11`, `-R11s7`, `-R9` and `-R1`.
2. Input: `disasterm3-mirror` (abrarmohammedtanzim).
3. Smoke test, then the full run:
   - Run All with `SMOKE_TEST = True` (8 images per model, a few minutes). If Cell 1 says RESTART THE KERNEL NOW, restart and Run All again.
   - Then set `SMOKE_TEST = False` and use Save & Run All (Commit), about 1 hour.
4. Outputs in `/kaggle/working/E5-step2/`: `per_image_counts.csv`, `reproduction.csv`, `clean_preview.csv`, `config.json` and `README.md`.

In [1]:
# ── Cell 1: Environment Setup (Run-All safe) ──────────────────────────────
# segmentation-models-pytorch 0.3.4 is needed to load the D4 checkpoints (their key layout).
import importlib.metadata as _md
import os, sys, subprocess

PINS = {
    "segmentation-models-pytorch": "0.3.4",   # baseline version (checkpoint key layout)
}

def _mismatched():
    out = []
    for pkg, want in PINS.items():
        try:
            have = _md.version(pkg)
        except _md.PackageNotFoundError:
            have = "not installed"
        if have != want:
            out.append(f"{pkg}: {have} → {want}")
    return out

if _mismatched():
    print("⏳ Installing pinned versions:")
    for _m in _mismatched():
        print(f"   {_m}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *(f"{k}=={v}" for k, v in PINS.items()), "huggingface_hub[hf_xet]"],
                   check=True)                # pip failure stops the run here
    if _mismatched():
        raise RuntimeError(f"Pins still wrong after install: {_mismatched()}")
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
        # background "Save & Run All": nothing imported yet, so new versions load on first import
        print("✓ Dependencies installed (background run) — continuing without restart.")
    else:
        raise SystemExit("✓ Dependencies installed. RESTART THE KERNEL NOW "
                         "(Run → Restart & clear cell outputs), then Run All again.")
else:
    print("✓ All pinned versions already installed — proceeding.")

import torch
print(f"  torch {torch.__version__}, python {sys.version.split()[0]}")

⏳ Installing pinned versions:
   segmentation-models-pytorch: not installed → 0.3.4
✓ Dependencies installed (background run) — continuing without restart.
  torch 2.11.0+cu128, python 3.13.15


In [2]:
# ── Cell 2: Configuration and inputs ──────────────────────────────────────
import glob
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import hf_hub_download

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"
MASK_TMP = "/tmp/combined_masks"    # where R11's cells write the 4-class masks

NUM_CLASSES = 4                     # 0=Background, 1=Intact, 2=Damaged, 3=Destroyed
CLASS_NAMES = ["Background", "Intact", "Damaged", "Destroyed"]
SMOKE_TEST = False                  # True = 8 validation images per model
OUT_DIR = Path("/kaggle/working/E5-step2")
HF_USER = "AbrarAlam"               # checkpoints: {HF_USER}/disasterm3-d4-<run>/final_model.pth

# in_channels: 6 = after-image then before-image (R11's recipe), 3 = after-image only.
# expected: (mIoU, Destroyed IoU) from each run's D4 CSV, standard and with 8-view TTA.
MODELS = {
    "R11":   {"in_channels": 6, "expected": {"std": (0.5811, 0.2911), "tta": (0.5875, 0.2965)}},
    "R11s7": {"in_channels": 6, "expected": {"std": (0.5867, 0.2869), "tta": (0.5913, 0.2928)}},
    "R9":    {"in_channels": 3, "expected": {"std": (0.5373, 0.2921), "tta": (0.5440, 0.3001)}},
    "R1":    {"in_channels": 3, "expected": {"std": (0.5374, 0.2686), "tta": (0.5440, 0.2822)}},
}
RUN_MODELS = ["R11", "R11s7", "R9", "R1"]
# The PC check of 26 Oct (xBD's original files): its 21 leaked validation pairs, by after-image stem (a preview only;
# step 1's list is applied on the PC)
PC_21 = ["socal_fire_00000081", "socal_fire_00000220", "socal_fire_00000222", "socal_fire_00000301",
         "socal_fire_00000341", "socal_fire_00000348", "socal_fire_00000436", "socal_fire_00000455",
         "socal_fire_00000497", "woolsey_fire_00000009", "woolsey_fire_00000030", "woolsey_fire_00000032",
         "woolsey_fire_00000474", "woolsey_fire_00000529", "woolsey_fire_00000573", "woolsey_fire_00000623",
         "woolsey_fire_00000632", "woolsey_fire_00000634", "woolsey_fire_00000703", "woolsey_fire_00000783",
         "woolsey_fire_00000810"]

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def find_checkpoint(name):
    """As in E1: a /kaggle/input folder named R11 / d4_R11 / R11-results, then Hugging Face."""
    wanted = {name.lower(), f"d4_{name}".lower(), f"{name}-results".lower()}
    hits = [p for p in glob.glob("/kaggle/input/**/*.pth", recursive=True)
            if any(part.lower() in wanted for part in Path(p).parts)]
    hits.sort(key=lambda p: Path(p).name != "final_model.pth")   # prefer final_model.pth
    if hits:
        return hits[0]
    repo = f"{HF_USER}/disasterm3-d4-{name}"
    errors = []
    for filename in ("final_model.pth", "last.pth"):   # last.pth holds the same epoch-40 weights
        try:
            return hf_hub_download(repo, filename, token=hf_token())
        except Exception as e:
            errors.append(f"{filename}: {type(e).__name__}")
    raise FileNotFoundError(f"No checkpoint for {name}: nothing under /kaggle/input and {repo} gave {errors}. "
                            f"Add the HF_TOKEN secret, or a Kaggle dataset with {name}/final_model.pth.")

assert MANIFEST_PATH.exists(), f"No manifest at {MANIFEST_PATH}: attach abrarmohammedtanzim/disasterm3-mirror"
CKPTS = {m: find_checkpoint(m) for m in RUN_MODELS}
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ E5 step 2{'  [SMOKE TEST]' if SMOKE_TEST else ''}: per-image counts for {RUN_MODELS}")
for m in RUN_MODELS:
    print(f"  {m} ({MODELS[m]['in_channels']} channels): {CKPTS[m]}")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}; outputs: {OUT_DIR}")

final_model.pth: reconstructing file:   0%|          |  0.00B /  130MB            

final_model.pth: downloading bytes:           |  0.00B            

final_model.pth: reconstructing file:   0%|          |  0.00B /  130MB            

final_model.pth: downloading bytes:           |  0.00B            

final_model.pth: reconstructing file:   0%|          |  0.00B /  130MB            

final_model.pth: downloading bytes:           |  0.00B            

final_model.pth: reconstructing file:   0%|          |  0.00B /  130MB            

final_model.pth: downloading bytes:           |  0.00B            

✓ E5 step 2: per-image counts for ['R11', 'R11s7', 'R9', 'R1']
  R11 (6 channels): /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R11/snapshots/31974dacfee2125a66639858433a113704edf9b3/final_model.pth
  R11s7 (6 channels): /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R11s7/snapshots/d0654a16a2b8b15d8c9989f22519d53cf4d403c0/final_model.pth
  R9 (3 channels): /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R9/snapshots/34897e1aa70742cac62a379561e1f969dccd4e10/final_model.pth
  R1 (3 channels): /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R1/snapshots/ebe008702466f2db4dfac0d1c51d269e2974f59a/final_model.pth
  Device: Tesla T4; outputs: /kaggle/working/E5-step2


## 1. Data: the same 645 validation images as every D4 run
Cell 3 is R11's (via R16c step 3), verbatim except that its mask folder is `MASK_TMP`. Cell 4 is R11's split. So the images match R1–R16c exactly.

In [3]:
# ── Cell 3: Load manifest and build the 4-class masks (R11's Cell 4, verbatim but for MASK_TMP) ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

# ── Filter: ONLY Building Damage Assessment segmentation entries ──
bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

# ── Path resolution (same logic as before) ──
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

# ── Group entries by underlying post-disaster image ──
by_image = defaultdict(dict)  # post_image_path -> {folder_type: mask_path}

for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":   # skip SAR duplicates, keep optical only
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name  # e.g. train_building_intact_mask
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img
        by_image[post_img]["pre_image_path"] = e.get("pre_image_path")   # R11: the before-image

print(f"Unique images with at least one mask: {len(by_image):,}")

# ── Build combined (image, mask) pairs ──
FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,   # use 2 here instead if you want to collapse into "damaged"
}

# ── FIX: Create temporary directory on Kaggle hard drive for masks ──
os.makedirs(MASK_TMP, exist_ok=True)

pairs = []
skipped_img = 0

for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id   # later classes overwrite earlier if they overlap

    if combined_mask is None:
        continue  # no valid masks found for this image at all

    # ── FIX: Save mask to hard drive instead of RAM ──
    mask_filename = Path(img_path).name
    save_path = f"{MASK_TMP}/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)

    pairs.append({
        "image_path": img_path,
        "mask_path": save_path,   # Store the string path, NOT the array!
        "pre_image_path": resolve_path(mask_dict.get("pre_image_path")),   # R11: None if not found
    })

print(f"\n✓ Built {len(pairs):,} valid (image, mask) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")
print(f"  Before-image not found: {sum(p['pre_image_path'] is None for p in pairs):,} (must be 0 for R11)")

Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask) pairs
  Skipped (image not found): 0
  Before-image not found: 0 (must be 0 for R11)


In [4]:
# ── Cell 4: The validation split (identical to every D4 run) and event names ──
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
if SMOKE_TEST:
    val_pairs = val_pairs[:8]
missing_pre = sum(p["pre_image_path"] is None for p in val_pairs)
assert missing_pre == 0, f"{missing_pre} pairs have no before-image"

XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")

def event_and_source(image_path):
    """Same rule as E1: 'hurricane_florence_00000261_post_disaster.png' -> ('hurricane_florence', 'xBD')."""
    name = Path(image_path).name
    m = XBD_NAME.match(name)
    if m:
        return m.group(1), "xBD"
    m = OTHER_NAME.match(name)
    return re.sub(r"\d+$", "", m.group(1) if m else Path(name).stem), "other"

events = [event_and_source(p["image_path"]) for p in val_pairs]
n_xbd = sum(1 for _, source in events if source == "xBD")
print(f"✓ Validation: {len(val_pairs)} images ({n_xbd} xBD, {len(val_pairs) - n_xbd} other). "
      "Expected 645 (560 xBD, 85 other) for the full run.")
if not SMOKE_TEST:
    assert (len(val_pairs), n_xbd) == (645, 560), "not the D4 split"

✓ Validation: 645 images (560 xBD, 85 other). Expected 645 (560 xBD, 85 other) for the full run.


## 2. Each model on the 645 validation images
- Cell 5 is the D4 prediction code from R16c step 3, unchanged except that the number of input channels follows the model: 6 for R11 and R11s7, 3 for R9 and R1.
- Cell 6 keeps one 4 × 4 count per image and mode, so any set of images can be removed later.

In [5]:
# ── Cell 5: The U-Nets (the D4 prediction code) and the E-protocol metrics ──
import segmentation_models_pytorch as smp
import torch.nn.functional as F

def read_triplet(pair):
    """After-image and before-image (RGB, same size) and the 4-class mask, as in the D4 dataset class."""
    post, pre = cv2.imread(pair["image_path"]), cv2.imread(pair["pre_image_path"])
    gt = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
    if post is None or pre is None or gt is None:
        return None
    post, pre = cv2.cvtColor(post, cv2.COLOR_BGR2RGB), cv2.cvtColor(pre, cv2.COLOR_BGR2RGB)
    h, w = post.shape[:2]
    if pre.shape != post.shape:
        pre = cv2.resize(pre, (w, h), interpolation=cv2.INTER_LINEAR)
    if gt.shape != (h, w):
        gt = cv2.resize(gt, (w, h), interpolation=cv2.INTER_NEAREST)
    return post, pre, gt

def load_unet(path, in_channels):
    """D4 checkpoint: U-Net + ResNet-50, 4 classes; 6 input channels (after-image, then before-image) or 3 (after only)."""
    state = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=in_channels, classes=NUM_CLASSES)
    model.load_state_dict(state)
    return model.to(DEVICE).eval()

IMNET_MEAN = np.array([0.485, 0.456, 0.406], np.float32) * 255
IMNET_STD = np.array([0.229, 0.224, 0.225], np.float32) * 255

def unet_input(post, pre, in_channels):
    """As the D4 val_transform: ImageNet normalisation; channels 0-2 the after-image, 3-5 the before-image (R11 only)."""
    parts = [(post.astype(np.float32) - IMNET_MEAN) / IMNET_STD]
    if in_channels == 6:
        parts.append((pre.astype(np.float32) - IMNET_MEAN) / IMNET_STD)
    x = np.concatenate(parts, 2)
    return torch.from_numpy(x.transpose(2, 0, 1).copy()).unsqueeze(0).to(DEVICE)

def _forward_once(model, x):
    """Logits at x's own HxW: pad to a multiple of 32, exactly as in the D4 notebooks."""
    h, w = x.shape[-2:]
    ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
    if ph or pw:
        x = F.pad(x, (0, pw, 0, ph), mode="reflect")
    return model(x)[..., :h, :w].float()

# The D4 notebooks' 8 dihedral views: rotation 0/90/180/270 × (no flip, horizontal flip)
TTA_VIEWS = [(k, flip) for flip in (False, True) for k in range(4)]

def _tta_fwd(x, k, flip):
    x = torch.rot90(x, k, dims=(2, 3))
    return torch.flip(x, dims=(3,)) if flip else x

def _tta_inv(y, k, flip):
    if flip:
        y = torch.flip(y, dims=(3,))
    return torch.rot90(y, -k, dims=(2, 3))

@torch.no_grad()
def predict_std_and_tta(model, x):
    """One pass over the 8 views (as in E1). The identity view (k=0, no flip) is the standard prediction."""
    prob_sum, pred_std = 0, None
    with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
        for k, flip in TTA_VIEWS:
            logits = _forward_once(model, _tta_fwd(x, k, flip))
            if k == 0 and not flip:
                pred_std = logits.argmax(1)
            prob_sum = prob_sum + torch.softmax(_tta_inv(logits, k, flip), dim=1)
    pred_tta = (prob_sum / len(TTA_VIEWS)).argmax(1)
    return (pred_std[0].cpu().numpy().astype(np.uint8),
            pred_tta[0].cpu().numpy().astype(np.uint8))

def confusion(gt, pred):
    """4×4 pixel counts, rows = truth, columns = prediction."""
    idx = gt.astype(np.int64).ravel() * NUM_CLASSES + pred.astype(np.int64).ravel()
    return np.bincount(idx, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)

def metrics(cm):
    """Dataset-level metrics from a confusion matrix (E1's formulas), plus building IoU (classes 1-3 merged)."""
    cm = cm.astype(np.float64)
    eps = 1e-8
    tp = np.diag(cm)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    iou = tp / (tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    building_tp = cm[1:, 1:].sum()
    building_iou = building_tp / (building_tp + cm[0, 1:].sum() + cm[1:, 0].sum() + eps)
    return {"miou": float(iou.mean()), "iou": iou, "precision": precision, "recall": recall,
            "building_iou": float(building_iou)}

_toy = np.array([[0, 1], [2, 3]], np.uint8)
assert (metrics(confusion(_toy, _toy))["iou"] > 0.99).all()
assert unet_input(np.zeros((4, 4, 3), np.uint8), np.zeros((4, 4, 3), np.uint8), 3).shape[1] == 3
assert unet_input(np.zeros((4, 4, 3), np.uint8), np.zeros((4, 4, 3), np.uint8), 6).shape[1] == 6
print("✓ Prediction code ready (self-checks passed)")

✓ Prediction code ready (self-checks passed)


In [6]:
# ── Cell 6: Each model on the validation images: one 4x4 count per image and mode ──
from tqdm.auto import tqdm

rows, model_minutes = [], {}
for name in RUN_MODELS:
    t0 = time.time()
    unet = load_unet(CKPTS[name], MODELS[name]["in_channels"])
    for pair, (event, source) in tqdm(list(zip(val_pairs, events)), desc=name):
        data = read_triplet(pair)
        if data is None:
            raise RuntimeError(f"Unreadable validation pair: {pair['image_path']}")
        post, pre, gt = data
        pred_std, pred_tta = predict_std_and_tta(unet, unet_input(post, pre, MODELS[name]["in_channels"]))
        for mode, pred in (("std", pred_std), ("tta", pred_tta)):
            cm = confusion(gt, pred)
            rows.append({"model": name, "mode": mode, "image": Path(pair["image_path"]).name, "event": event,
                         "source": source,
                         **{f"t{i}_p{j}": int(cm[i, j]) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES)}})
    del unet
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    model_minutes[name] = round((time.time() - t0) / 60, 1)
    print(f"✓ {name}: {len(val_pairs)} images in {model_minutes[name]} min")

R11:   0%|          | 0/645 [00:00<?, ?it/s]

✓ R11: 645 images in 13.3 min


R11s7:   0%|          | 0/645 [00:00<?, ?it/s]

✓ R11s7: 645 images in 6.3 min


R9:   0%|          | 0/645 [00:00<?, ?it/s]

✓ R9: 645 images in 5.7 min


R1:   0%|          | 0/645 [00:00<?, ?it/s]

✓ R1: 645 images in 5.7 min


In [7]:
# ── Cell 7: Results: each model against its D4 CSV, a preview without the leaks, and the output files ──
import pandas as pd

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

CM_COLS = [f"t{i}_p{j}" for i in range(NUM_CLASSES) for j in range(NUM_CLASSES)]
per_image = pd.DataFrame(rows)
per_image.to_csv(OUT_DIR / "per_image_counts.csv", index=False)

def score(df):
    """(mIoU, Destroyed IoU) of the summed counts of these rows."""
    m = metrics(df[CM_COLS].to_numpy().sum(0).reshape(NUM_CLASSES, NUM_CLASSES))
    return m["miou"], float(m["iou"][3])

pc21 = {f"{s}_post_disaster.png" for s in PC_21}
fire = per_image["image"].str.startswith(("socal_fire_", "woolsey_fire_"))
repro, preview = [], []
for name in RUN_MODELS:
    for mode in ("std", "tta"):
        sel = (per_image["model"] == name) & (per_image["mode"] == mode)
        miou, dest = score(per_image[sel])
        exp = MODELS[name]["expected"][mode]
        ok = ("n/a (smoke test)" if SMOKE_TEST else
              "yes" if abs(miou - exp[0]) < 5e-4 and abs(dest - exp[1]) < 5e-4 else "NO")
        repro.append({"model": name, "mode": mode, "miou": round(miou, 4), "destroyed_iou": round(dest, 4),
                      "d4_miou": exp[0], "d4_destroyed_iou": exp[1], "reproduces_d4": ok})
        m21, d21 = score(per_image[sel & ~per_image["image"].isin(pc21)])
        mfire, dfire = score(per_image[sel & ~fire])
        preview.append({"model": name, "mode": mode, "images": int(sel.sum()),
                        "destroyed_iou_all": round(dest, 4), "minus_pc21": round(d21, 4),
                        "minus_86_fire": round(dfire, 4), "miou_all": round(miou, 4),
                        "miou_minus_pc21": round(m21, 4), "miou_minus_86_fire": round(mfire, 4)})
repro_df, preview_df = pd.DataFrame(repro), pd.DataFrame(preview)
repro_df.to_csv(OUT_DIR / "reproduction.csv", index=False)
preview_df.to_csv(OUT_DIR / "clean_preview.csv", index=False)
n_images = int(((per_image["model"] == RUN_MODELS[0]) & (per_image["mode"] == "std")).sum())
n_fire = int((fire & (per_image["model"] == RUN_MODELS[0]) & (per_image["mode"] == "std")).sum())
n_pc21 = int((per_image["image"].isin(pc21) & (per_image["model"] == RUN_MODELS[0]) & (per_image["mode"] == "std")).sum())
print(repro_df.to_string(index=False))
print()
print(preview_df.to_string(index=False))
all_ok = SMOKE_TEST or all(r["reproduces_d4"] == "yes" for r in repro)
print(f"\nImages per model: {n_images}; of them fire ({n_fire}, expected 86) and the PC check's pairs ({n_pc21}, expected 21)")
print("✓ Every model reproduces its D4 CSV" if all_ok and not SMOKE_TEST else
      "Smoke test: reproduction not checked" if SMOKE_TEST else
      "⚠ A model doesn't reproduce its D4 CSV: check its checkpoint before using its clean scores")

config = {"task": "E5 step 2", "smoke_test": SMOKE_TEST, "val_images": n_images, "models": RUN_MODELS,
          "checkpoints": {m: str(CKPTS[m]) for m in RUN_MODELS}, "in_channels": {m: MODELS[m]["in_channels"] for m in RUN_MODELS},
          "expected": {m: MODELS[m]["expected"] for m in RUN_MODELS}, "minutes": model_minutes,
          "all_reproduce_d4": all_ok, "fire_images": n_fire, "pc21_images": n_pc21}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")

(OUT_DIR / "README.md").write_text("\n".join([
    "# E5 step 2: per-image pixel counts for R11, R11s7, R9 and R1", "",
    f"{n_images} validation images{' (SMOKE TEST)' if SMOKE_TEST else ''}, the D4 split. Each model as the D4 notebooks "
    "ran it: full size, padded to a multiple of 32, standard and 8-view TTA. R11 and R11s7 read the after- and "
    "before-image (6 channels), R9 and R1 the after-image only.", "",
    "## Each model against its D4 CSV", "",
    as_markdown(repro_df), "",
    "## Preview: Destroyed IoU and mIoU without leaked pairs", "",
    "`minus_pc21` drops the 21 pairs the PC check found (xBD's original files); `minus_86_fire` drops every "
    "socal_fire and woolsey_fire image, an upper bound on the effect. The exact list comes from E5 step 1 and is "
    "applied on the PC.", "",
    as_markdown(preview_df), "",
    "## How it was scored, and the alternatives", "",
    "- A 4 × 4 pixel count (truth × prediction) per image, model and mode. A score on any subset is the sum of its "
    "images' counts, so dropping leaked pairs needs no new GPU run. Summed over all 645 images, the counts must "
    "reproduce each D4 CSV, which checks that nothing in the prediction changed.",
    "- Alternative: re-run inference for each leak list. It gives the same numbers, but needs a GPU run each time "
    "the list changes (21, 40, step 1's).",
    "- Alternative: re-score only from the leaked images' pixels, as was done for R16c from its per-building file. "
    "That works for R16c, which saved per-building counts, and not for these runs, which saved only totals.", "",
    "## Files", "",
    "- `per_image_counts.csv`: model, mode, image, event, source and the 16 counts `t<truth>_p<prediction>`",
    "- `reproduction.csv`, `clean_preview.csv`, `config.json`",
]), encoding="utf-8")
print(f"\n✓ Wrote {OUT_DIR}: per_image_counts.csv, reproduction.csv, clean_preview.csv, config.json, README.md")

model mode   miou  destroyed_iou  d4_miou  d4_destroyed_iou reproduces_d4
  R11  std 0.5811         0.2911   0.5811            0.2911           yes
  R11  tta 0.5875         0.2965   0.5875            0.2965           yes
R11s7  std 0.5867         0.2869   0.5867            0.2869           yes
R11s7  tta 0.5913         0.2928   0.5913            0.2928           yes
   R9  std 0.5373         0.2921   0.5373            0.2921           yes
   R9  tta 0.5440         0.3001   0.5440            0.3001           yes
   R1  std 0.5374         0.2686   0.5374            0.2686           yes
   R1  tta 0.5440         0.2822   0.5440            0.2822           yes

model mode  images  destroyed_iou_all  minus_pc21  minus_86_fire  miou_all  miou_minus_pc21  miou_minus_86_fire
  R11  std     645             0.2911      0.2771         0.2524    0.5811           0.5776              0.5718
  R11  tta     645             0.2965      0.2824         0.2574    0.5875           0.5839              0.57